## About this NoteBook

* Contains RoBerta Model for SemEval Classification.
* The labels are multiclass.
* The dataset is raw and unaugmented.

In [1]:
#Importing the Libraries

!pip install simpletransformers
!pip install transformers-interpret

import pandas
import numpy
from ast import literal_eval
from sklearn.metrics import f1_score, classification_report
from simpletransformers.classification import MultiLabelClassificationModel
from transformers_interpret import SequenceClassificationExplainer

In [20]:
trainPath = "../input/raw-data/SE - RawTrain - T1T2 - Emo.csv"
testPath = "../input/raw-data/SE - RawTest - T1T2 - Emo.csv"

perList = []
train_df = pandas.read_csv(trainPath)

for label in train_df['label']:
    perList.extend(literal_eval(label))
perList = list(set(perList))

In [29]:
train_df = pandas.read_csv(trainPath)
train_df.dropna(inplace=True)

trainCol = []
for label in train_df['label']:
    tempL = [0] * len(perList)
    for indiLab in literal_eval(label):
        tempL[perList.index(indiLab)] = 1
    trainCol.append(tempL)
train_df['label_desc'] = train_df['label']
train_df['label'] = trainCol
train_df['sentence'] = train_df['sentence']  + ' ' + train_df['emotions']
train_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)

newCol = []
for text in train_df['text']:
    newCol.append(text.replace('neutral', ''))
train_df['text'] = newCol

train_df.head()

In [30]:
test_df = pandas.read_csv(testPath)
test_df.dropna(inplace=True)

testCol = []
for label in test_df['label']:
    tempL = [0] * len(perList)
    for indiLab in literal_eval(label):
        tempL[perList.index(indiLab)] = 1
    testCol.append(tempL)
test_df['label_desc'] = test_df['label']
test_df['label'] = testCol
test_df['sentence'] = test_df['sentence']  + ' ' + test_df['emotions']
test_df.rename({'sentence':'text', 'label':'labels'}, axis=1, inplace=True)

newCol = []
for text in test_df['text']:
    newCol.append(text.replace('neutral', ''))
test_df['text'] = newCol

test_df.head()

In [31]:
# Training Roberta Transformer on GoEmotions as MultiLabel Classification

model = MultiLabelClassificationModel('roberta', 'roberta-base', num_labels=len(perList), 
                                      args={'train_batch_size':4, 'fp16': False, 'overwrite_output_dir': True,                                         
                                            'gradient_accumulation_steps':8, 'learning_rate': 3e-5,
                                            'num_train_epochs': 2, "reprocess_input_data": True, 
                                            'max_seq_length': 16})

model.train_model(train_df)
result, model_outputs, wrong_predictions = model.eval_model(test_df)
predictions, raw_outputs = model.predict(list(test_df['text']))

print('-'*100)

print(f"Accuracy & Loss - {result}")
print(f"F1-Micro Score - {f1_score(testCol, predictions, average='micro')}")
print(f"F1-Macro Score - {f1_score(testCol, predictions, average='macro')}")

In [36]:
idx = 16
cls_explainer = SequenceClassificationExplainer(model.model, model.tokenizer, custom_labels=perList)
word_attributions = cls_explainer(test_df.iloc[idx]['text'])
label = perList[int(cls_explainer.predicted_class_index)]
word_attributions

In [37]:
print(f"Text : {test_df.iloc[idx]['text']}")
print('-'*100)
print(f"True Label : {test_df.iloc[idx]['label_desc']}")
print(f"Predicted Label : {label}")